# Read RollCall Transcript

Fetches and extracts text from the Trump/Llamas NBC interview transcript.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import trafilatura
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel
from htmlmin import minify
from bs4 import BeautifulSoup
import ast

In [2]:
url = "https://rollcall.com/factbase/trump/transcript/donald-trump-interview-tom-llamas-nbc-news-february-4-2026/"

In [ ]:
def get_html(url: str) -> str | None:
    return trafilatura.fetch_url(url)


def get_main_content_html(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["head", "script", "style", "nav", "footer", "header"]):
        tag.decompose()

    body_html = str(soup.body or soup)
    return body_html[:30000]

class ExtractorScript(BaseModel):
    code: str


llm = init_chat_model("claude-sonnet-4-5")
structured_llm = llm.with_structured_output(ExtractorScript)

def get_extractor_script(html_sample: str) -> str:
    code = structured_llm.invoke([

        SystemMessage(content="""You are an expert web scraper.
        Write self-contained Python with all imports. BeautifulSoup only. 
        Write concise, readable code. Prefer clarity over cleverness.
        Define a function named `extract(html: str) -> str`.
        Your only job is to identify and extract the primary content structure."""),

        HumanMessage(content=f"""
        Analyze this HTML and write an `extract` function tailored to its specific structure.
        Preserve meaningful information and output clean text.
        HTML: {html_sample}"""),

    ]).code
    ast.parse(code)
    return code


def scrape(url: str) -> str:

    html = get_html(url)
    html_sample = get_main_content_html(html)
    code = get_extractor_script(html_sample)
    return code

    namespace = {}
    exec(code, namespace)
    
    return namespace["extract"](html)


In [42]:
text = scrape(url)

In [43]:
print(text)


from bs4 import BeautifulSoup
import re

def extract(html: str) -> str:
    """Extract transcript content from Factbase transcript page."""
    soup = BeautifulSoup(html, 'html.parser')
    lines = []
    
    # Extract title
    title_elem = soup.find('h1', class_='text-center')
    if title_elem:
        title = title_elem.get_text(strip=True)
        lines.append(title)
        lines.append('=' * len(title))
        lines.append('')
    
    # Extract speaker information
    subject = soup.find('span', class_='font-graphik font-medium text-xl text-black')
    if subject:
        lines.append(f"Subject: {subject.get_text(strip=True)}")
        lines.append('')
    
    # Extract topics
    topics_section = soup.find('div', attrs={'x-show': "currentTab === 'topics'"})
    if topics_section:
        topic_links = topics_section.find_all('a', class_='text-[#015582]')
        if topic_links:
            lines.append('Topics:')
            for link in topic_links:
                topic_t

In [46]:
from bs4 import BeautifulSoup
import re

def extract(html: str) -> str:
    """Extract transcript content from Factbase transcript page."""
    soup = BeautifulSoup(html, 'html.parser')
    lines = []
    
    # Extract title
    title_elem = soup.find('h1', class_='text-center')
    if title_elem:
        title = title_elem.get_text(strip=True)
        lines.append(title)
        lines.append('=' * len(title))
        lines.append('')
    
    # Extract speaker information
    subject = soup.find('span', class_='font-graphik font-medium text-xl text-black')
    if subject:
        lines.append(f"Subject: {subject.get_text(strip=True)}")
        lines.append('')
    
    # Extract topics
    topics_section = soup.find('div', attrs={'x-show': "currentTab === 'topics'"})
    if topics_section:
        topic_links = topics_section.find_all('a', class_='text-[#015582]')
        if topic_links:
            lines.append('Topics:')
            for link in topic_links:
                topic_text = link.get_text(strip=True)
                if topic_text:
                    lines.append(f"  - {topic_text}")
            lines.append('')
    
    # Extract entities
    entities_section = soup.find('div', attrs={'x-show': "currentTab === 'entities'"})
    if entities_section:
        entity_divs = entities_section.find_all('div', class_='text-[#015582]')
        if entity_divs:
            lines.append('Entities:')
            for div in entity_divs:
                entity = div.get_text(strip=True)
                if entity:
                    lines.append(f"  - {entity}")
            lines.append('')
    
    # Extract speaker statistics
    speakers_section = soup.find('div', attrs={'x-show': "currentTab === 'speakers'"})
    if speakers_section:
        lines.append('Speakers:')
        speaker_blocks = speakers_section.find_all('div', class_='flex-1 h-content')
        for block in speaker_blocks:
            speaker_name_elem = block.find('div', class_='font-graphik text-sm font-medium')
            if speaker_name_elem:
                speaker_name = speaker_name_elem.get_text(strip=True)
                lines.append(f"\n  {speaker_name}:")
                
                # Extract statistics
                stats = block.find_all('div', class_='font-graphik text-xs font-medium text-[#2F3C4B]')
                for stat in stats:
                    stat_text = stat.get_text(strip=True)
                    if stat_text:
                        lines.append(f"    {stat_text}")
        lines.append('')
    
    # Extract transcript entries
    lines.append('FULL TRANSCRIPT')
    lines.append('-' * 50)
    lines.append('')
    
    transcript_entries = soup.find_all('div', class_='mb-4 border-b mx-6 my-4')
    
    for entry in transcript_entries:
        # Find speaker and timestamp
        header = entry.find('div', class_='mb-2 flex')
        if header:
            speaker_elem = header.find('h2', class_='text-md inline')
            timestamp_elem = header.find('span', class_='text-xs text-gray-600')
            
            if speaker_elem:
                speaker = speaker_elem.get_text(strip=True)
                timestamp = ''
                if timestamp_elem:
                    timestamp = timestamp_elem.get_text(strip=True)
                
                lines.append(f"[{speaker}] {timestamp}")
        
        # Find content
        content_elem = entry.find('div', class_='flex-auto text-md text-gray-600 leading-loose')
        if content_elem:
            content = content_elem.get_text(strip=True)
            if content:
                lines.append(content)
                lines.append('')
    
    return '\n'.join(lines)

print(extract(get_html(
    'https://rollcall.com/factbase/trump/transcript/donald-trump-speech-military-families-fort-bragg-february-13-2026/'
    )))

Speech: Donald Trump Addresses Military Families at Fort Bragg, North Carolina - February 13, 2026

Subject: Donald J. Trump

Topics:
  - Conflict, War and Peace
  - Politics > Government
  - Politics
  - Conflict, War and Peace > Armed Conflict
  - Politics > Government > Defence > Military Equipment > Military Weaponry
  - Politics > Government > Defence > Armed Forces

Entities:
  - best
  - country
  - Donald J. Trump
  - forces
  - investor
  - Michael Whatley
  - military
  - part
  - reason
  - Richard Hudson (American politician)

Speakers:

FULL TRANSCRIPT
--------------------------------------------------

[Melania Trump] 
Good afternoon. Hello, and thank you for your warm welcome. The president and I are grateful to be with you to honor the passion you share through unmatched bravery, resilience and service to our great nation. Thank you for defending our freedom. It is with immense pleasure for me to address the Fort Bragg and Pope Army Airfield community today.

[Melania T

In [39]:
url

'https://rollcall.com/factbase/trump/transcript/donald-trump-interview-tom-llamas-nbc-news-february-4-2026/'